# SwissProt STRING Interactions

This notebook processes STRING database protein-protein interaction (PPI) data and adds it to the SwissProt reasoning dataset.

## Configuration

**Important**: Update the paths in the configuration cell below to match your local setup.

In [ ]:
# ============================================================================
# CONFIGURATION - Update these paths to match your local setup
# ============================================================================
import os

# Base directory for SwissProt dataset
SWISSPROT_DATA_DIR = os.path.join("data", "swissprot")  # Update this path

# STRING database files directory
STRING_DATA_DIR = os.path.join("data", "string")  # Update this path

# Output directory for processed files
OUTPUT_DIR = os.path.join("data", "processed")  # Update this path

# Paths to STRING database files
UNIPROT_STRING_MAPPING = os.path.join(STRING_DATA_DIR, "all_organisms.uniprot_2_string.tsv")
STRING_PROTEIN_INFO = os.path.join(STRING_DATA_DIR, "protein.info.v12.0.txt")
STRING_PROTEIN_LINKS_PARQUET = os.path.join(STRING_DATA_DIR, "parquet_output_duckdb", "protein_links_space_delimited.parquet")
STRING_PROTEIN_LINKS_GZ = os.path.join(STRING_DATA_DIR, "protein.links.detailed.v12.0.txt.gz")

# Create output directory if it doesn't exist
os.makedirs(OUTPUT_DIR, exist_ok=True)

print("Configuration loaded. Update paths above if needed.")

Run the following on compute node

In [2]:
python -m venv $SLURM_TMPDIR/bioreason_env
module load python
module load gcc arrow
module load postgresql

source $SLURM_TMPDIR/bioreason_env/bin/activate
pip install --no-index notebook duckdb polars-u64-idx datasets
jupyter notebook --no-browser --ip=$(hostname -f) --port=8888

Downloads (from https://string-db.org/cgi/download)

https://stringdb-downloads.org/download/protein.links.detailed.v12.0.txt.gz

https://stringdb-downloads.org/download/protein.info.v12.0.txt.gz

# Initial Analysis

In [1]:
from datasets import load_dataset

# Load the train split
train_dataset = load_dataset(
    "wanglab/cafa5",
    data_files="swissprot_reasoning/train-*.parquet",
    split="train"
)

# Load the test split
test_dataset = load_dataset(
    "wanglab/cafa5",
    data_files="swissprot_reasoning/test-*.parquet",
    split="train"
)

print(train_dataset)
print(test_dataset)


Dataset({
    features: ['protein_id', 'protein_names', 'protein_function', 'organism', 'length', 'subcellular_location', 'sequence', 'go_ids', 'go_bp', 'go_mf', 'go_cc', 'structure_path', 'interpro_ids', 'interpro_location'],
    num_rows: 333677
})
Dataset({
    features: ['protein_id', 'protein_names', 'protein_function', 'organism', 'length', 'subcellular_location', 'sequence', 'go_ids', 'go_bp', 'go_mf', 'go_cc', 'structure_path', 'interpro_ids', 'interpro_location'],
    num_rows: 83420
})


Load both splits of the cafa5 from storage

In [2]:
# Convert HuggingFace Datasets to Polars DataFrames
import pandas as pd
import polars as pl

# Convert train and test datasets to pandas DataFrames, then to polars
train_df = train_dataset.to_pandas()
test_df = test_dataset.to_pandas()

train_pl = pl.from_pandas(train_df)
test_pl = pl.from_pandas(test_df)

# concatenate into one DataFrame
swissprot_pl = pl.concat([train_pl, test_pl], how="vertical")

# sanity check
print(swissprot_pl.schema)
print(swissprot_pl.head(3))




Schema([('protein_id', String), ('protein_names', String), ('protein_function', String), ('organism', String), ('length', Int64), ('subcellular_location', String), ('sequence', String), ('go_ids', List(String)), ('go_bp', List(String)), ('go_mf', List(String)), ('go_cc', List(String)), ('structure_path', String), ('interpro_ids', List(String)), ('interpro_location', String)])
shape: (3, 14)
┌───────────┬───────────┬───────────┬───────────┬───┬───────────┬───────────┬───────────┬──────────┐
│ protein_i ┆ protein_n ┆ protein_f ┆ organism  ┆ … ┆ go_cc     ┆ structure ┆ interpro_ ┆ interpro │
│ d         ┆ ames      ┆ unction   ┆ ---       ┆   ┆ ---       ┆ _path     ┆ ids       ┆ _locatio │
│ ---       ┆ ---       ┆ ---       ┆ str       ┆   ┆ list[str] ┆ ---       ┆ ---       ┆ n        │
│ str       ┆ str       ┆ str       ┆           ┆   ┆           ┆ str       ┆ list[str] ┆ ---      │
│           ┆           ┆           ┆           ┆   ┆           ┆           ┆           ┆ str      │


In [29]:
test_pl

protein_id,protein_names,protein_function,organism,length,subcellular_location,sequence,go_ids,go_bp,go_mf,go_cc,structure_path,interpro_ids,interpro_location
str,str,str,str,i64,str,str,list[str],list[str],list[str],list[str],str,list[str],str
"""B2IUL7""","""Light-independent protochlorop…","""Component of the dark-operativ…","""Nostoc punctiforme (strain ATC…",288,null,"""MKLAVYGKGGIGKSTTSCNISVALAKRGKK…","[""GO:0008150"", ""GO:0003674"", … ""GO:0005524""]","[""GO:0008150"", ""GO:0008152"", … ""GO:0036068""]","[""GO:0003674"", ""GO:0005488"", … ""GO:0005524""]",null,"""AF-B2IUL7-F1-model_v4.cif""","[""IPR005971"", ""IPR030655"", … ""IPR000392""]","""{""IPR027417"": [1, 268], ""IPR00…"
"""Q1RJY6""","""Formamidopyrimidine-DNA glycos…","""Involved in base excision repa…","""Rickettsia bellii (strain RML3…",273,null,"""MPELPEVETLKNSLESKLIGLVIKKVEFKR…","[""GO:0008150"", ""GO:0003674"", … ""GO:0006284""]","[""GO:0008150"", ""GO:0008152"", … ""GO:0006284""]","[""GO:0003674"", ""GO:0005488"", … ""GO:0008534""]",null,"""AF-Q1RJY6-F1-model_v4.cif""","[""IPR015887"", ""IPR020629"", … ""IPR000214""]","""{""IPR035937"": [1, 141], ""IPR01…"
"""P31614""","""Hemagglutinin-esterase""","""Structural protein that makes …","""Murine coronavirus (strain S)""",439,"""Virion membrane; Host cell mem…","""MGCMCIAMAPRTLLLLIGCQLVFGFNEPLN…","[""GO:0008150"", ""GO:0005575"", … ""GO:0106330""]","[""GO:0008150"", ""GO:0009987"", … ""GO:0019064""]","[""GO:0003674"", ""GO:0005488"", … ""GO:0106330""]","[""GO:0005575"", ""GO:0110165"", … ""GO:0020002""]",null,"[""IPR042545"", ""IPR003860"", … ""IPR007142""]","""{""IPR008980"": [145, 294], ""IPR…"
"""Q92FZ5""","""Transcription elongation facto…","""Necessary for efficient RNA po…","""Rickettsia conorii (strain ATC…",162,null,"""MNTKFPITAKGFEKLEHELKHLKHVERKKI…","[""GO:0008150"", ""GO:0003674"", … ""GO:0032784""]","[""GO:0008150"", ""GO:0065007"", … ""GO:0032784""]","[""GO:0003674"", ""GO:0005488"", … ""GO:0070063""]",null,"""AF-Q92FZ5-F1-model_v4.cif""","[""IPR018151"", ""IPR036953"", … ""IPR028624""]","""{""IPR036805"": [1, 78], ""IPR036…"
"""Q8UE43""","""Dihydroxy-acid dehydratase""","""Functions in the biosynthesis …","""Agrobacterium fabrum (strain C…",611,null,"""MPAYRSRTTTHGRNMAGARGLWRATGMKDS…","[""GO:0008150"", ""GO:0003674"", … ""GO:0009097""]","[""GO:0008150"", ""GO:0008152"", … ""GO:0009097""]","[""GO:0003674"", ""GO:0005488"", … ""GO:0004160""]",null,"""AF-Q8UE43-F1-model_v4.cif""","[""IPR000581"", ""IPR056740"", … ""IPR020558""]","""{""IPR037237"": [4, 420], ""IPR04…"
…,…,…,…,…,…,…,…,…,…,…,…,…,…
"""Q0C5R4""","""3-phosphoshikimate 1-carboxyvi…","""Catalyzes the transfer of the …","""Hyphomonas neptunium (strain A…",439,"""Cytoplasm""","""MVWTSHPVKRLAGAIRAPGDKSCSHRALIF…","[""GO:0008150"", ""GO:0005575"", … ""GO:0009073""]","[""GO:0008150"", ""GO:0008152"", … ""GO:0009073""]","[""GO:0003674"", ""GO:0003824"", … ""GO:0003866""]","[""GO:0005575"", ""GO:0110165"", ""GO:0005737""]","""AF-Q0C5R4-F1-model_v4.cif""","[""IPR013792"", ""IPR006264"", … ""IPR036968""]","""{""IPR013792"": [5, 436], ""IPR03…"
"""P0A6S8""","""Glycerol-3-phosphate dehydroge…",null,"""Escherichia coli O6:H1 (strain…",339,"""Cytoplasm""","""MNQRNASMTVIGAGSYGTALAITLARNGHE…","[""GO:0008150"", ""GO:0005575"", … ""GO:0046168""]","[""GO:0008150"", ""GO:0008152"", … ""GO:0046168""]","[""GO:0003674"", ""GO:0005488"", … ""GO:0051287""]","[""GO:0005575"", ""GO:0032991"", … ""GO:0009331""]","""AF-P0A6S8-F1-model_v4.cif""","[""IPR036291"", ""IPR013328"", … ""IPR011128""]","""{""IPR036291"": [8, 182], ""IPR00…"
"""B7NN18""","""Periplasmic nitrate reductase""","""Catalytic subunit of the perip…","""Escherichia coli O7:K1 (strain…",828,"""Periplasm""","""MKLSRRSFMKANAVAAAAAAAGLSVPGVAR…","[""GO:0008150"", ""GO:0005575"", … ""GO:0006777""]","[""GO:0008150"", ""GO:0008152"", … ""GO:0006777""]","[""GO:0003674"", ""GO:0005488"", … ""GO:0030151""]","[""GO:0005575"", ""GO:0032991"", … ""GO:1990204""]","""AF-B7NN18-F1-model_v4.cif""","[""IPR006656"", ""IPR

Load the uniprot to string conversion table (from string database)

In [3]:
# load uniprot to string data

import polars as pl

mapping_path = UNIPROT_STRING_MAPPING
cols = ["species", "uniprot_ac|uniprot_id", "string_id", "identity", "bit_score"]

# Load and clean, skipping the bad header
uniprot_string_map_df = pl.read_csv(
    mapping_path,
    separator="\t",
    comment_prefix="#",
    has_header=False,
    new_columns=cols,
    ignore_errors=True
)

# Split the combined column into accession and ID, then drop the original
uniprot_string_map_df = (
    uniprot_string_map_df
    .with_columns(
        pl.col("uniprot_ac|uniprot_id")
          .str.split_exact("|", 1)
          .struct
          .rename_fields(["uniprot_ac", "uniprot_id"])
          .alias("split")
    )
    .unnest("split")
    .drop("uniprot_ac|uniprot_id")
)

# Preview
# print(uniprot_string_map_df.head(5))


Reads the string/name table from raw text to polars

In [4]:
import polars as pl

# --- Configuration fallback: only set if not already defined ---
if 'protein_info_path' not in globals():
    protein_info_path = STRING_PROTEIN_INFO

# Define the actual column names (strip the leading '#')
cols = ["string_protein_id", "preferred_name", "protein_size", "annotation"]

# --- Load the uncompressed protein.info file, using our header names ---
protein_info_df = pl.read_csv(
    protein_info_path,
    separator="\t",
    has_header=False,   # file's first line isn't a usable header
    skip_rows=1,        # skip the '#...' header line
    new_columns=cols,    # apply our cleaned column names
    ignore_errors=True,
)

# Preview columns & first few rows
print("Columns:", protein_info_df.columns)
print(protein_info_df.head(5))


Columns: ['string_protein_id', 'preferred_name', 'protein_size', 'annotation']
shape: (5, 4)
┌───────────────────┬────────────────┬──────────────┬─────────────────────────────────┐
│ string_protein_id ┆ preferred_name ┆ protein_size ┆ annotation                      │
│ ---               ┆ ---            ┆ ---          ┆ ---                             │
│ str               ┆ str            ┆ i64          ┆ str                             │
╞═══════════════════╪════════════════╪══════════════╪═════════════════════════════════╡
│ 23.BEL05_00025    ┆ BEL05_00025    ┆ 429          ┆ Hypothetical protein; Incomple… │
│ 23.BEL05_00030    ┆ OEG73659.1     ┆ 203          ┆ 2-keto-4-pentenoate hydratase;… │
│ 23.BEL05_00035    ┆ OEG73660.1     ┆ 1140         ┆ Hybrid sensor histidine kinase… │
│ 23.BEL05_00040    ┆ acsA           ┆ 650          ┆ acetate--CoA ligase; Catalyzes… │
│ 23.BEL05_00045    ┆ OEG73662.1     ┆ 579          ┆ ATP-dependent helicase; Derive… │
└───────────────────┴──────

sys:1: UserWarning: CSV malformed: expected 1 rows, actual 54947921 rows, in chunk starting at byte offset 402138335, length 6031279343


In [5]:
protein_info_df.shape

(59309604, 4)

Brings the string ids into the cafa5 dataset, matching based on the uniprot id

In [6]:
# join uniprot cafa5 data with string id

import polars as pl

# 2) Prepare the mapping: only uniprot_ac → string_id
map_pl = uniprot_string_map_df.select(["uniprot_ac", "string_id"])

# 3) Join on protein_id ↔ uniprot_ac
joined = swissprot_pl.join(
    map_pl,
    left_on="protein_id",
    right_on="uniprot_ac",
    how="left"
)

swissprot_pl = joined.select(
    swissprot_pl.columns + ["string_id"]
)

# 5) Quick check
print(swissprot_pl.head(10))


shape: (10, 15)
┌───────────┬───────────┬───────────┬───────────┬───┬───────────┬───────────┬───────────┬──────────┐
│ protein_i ┆ protein_n ┆ protein_f ┆ organism  ┆ … ┆ structure ┆ interpro_ ┆ interpro_ ┆ string_i │
│ d         ┆ ames      ┆ unction   ┆ ---       ┆   ┆ _path     ┆ ids       ┆ location  ┆ d        │
│ ---       ┆ ---       ┆ ---       ┆ str       ┆   ┆ ---       ┆ ---       ┆ ---       ┆ ---      │
│ str       ┆ str       ┆ str       ┆           ┆   ┆ str       ┆ list[str] ┆ str       ┆ str      │
╞═══════════╪═══════════╪═══════════╪═══════════╪═══╪═══════════╪═══════════╪═══════════╪══════════╡
│ Q8CX00    ┆ Chaperoni ┆ Together  ┆ Streptoco ┆ … ┆ AF-Q8CX00 ┆ ["IPR0183 ┆ {"IPR0274 ┆ 208435.S │
│           ┆ n GroEL   ┆ with its  ┆ ccus agal ┆   ┆ -F1-model ┆ 70", "IPR ┆ 13": [5,  ┆ AG2074   │
│           ┆           ┆ co-chaper ┆ actiae    ┆   ┆ _v4.cif   ┆ 001844",  ┆ 519],     ┆          │
│           ┆           ┆ oni…      ┆ serot…    ┆   ┆           ┆ … "…     

Convert the String data file from txt to a parquet file

In [ ]:
# # This cell efficiently converts the STRING data file to a Parquet file using DuckDB.
# # MEMORY OPTIMIZED VERSION for large files (190GB+) on systems with limited RAM

# import duckdb
# import os
# import psutil

# # Paths
# input_path = '<path_to_string_data>/protein.links.detailed.v12.0.txt.gz'
# output_dir = '<path_to_string_data>/parquet_output_duckdb'
# output_path = os.path.join(output_dir, 'protein_links_space_delimited.parquet')

# # Ensure output directory exists
# os.makedirs(output_dir, exist_ok=True)

# # Check available memory and disk space
# available_memory_gb = psutil.virtual_memory().available / (1024**3)
# file_size_gb = os.path.getsize(input_path) / (1024**3)
# print(f"File size: {file_size_gb:.1f} GB")
# print(f"Available memory: {available_memory_gb:.1f} GB")

# print("Starting memory-optimized conversion with DuckDB...")

# # MEMORY OPTIMIZATION SETTINGS
# # Limit memory usage to ~60% of available RAM to leave room for OS and other processes
# max_memory_gb = max(1, int(available_memory_gb * 0.6))
# print(f"Setting DuckDB memory limit to {max_memory_gb} GB")

# # Reduce thread count for memory-constrained environments
# thread_count = min(8, os.cpu_count())  # Use fewer threads to reduce memory pressure
# print(f"Using {thread_count} threads")

# # Connect to DuckDB with optimized configuration
# # Using a temporary database file allows DuckDB to spill to disk when needed
# temp_db_path = os.path.join(output_dir, 'temp_conversion.duckdb')
# config = {
#     'memory_limit': f'{max_memory_gb}GB',
#     'threads': thread_count,
#     'temp_directory': '/tmp',
#     'preserve_insertion_order': False,
#     'enable_external_access': True
# }

# con = duckdb.connect(temp_db_path, config=config)

# try:
    
#     # Process the file with memory-efficient settings
#     print("Converting file (this may take longer but uses less memory)...")
#     con.execute(f"""
#         COPY (
#             SELECT *
#             FROM read_csv(
#                 '{input_path}', 
#                 header=True, 
#                 delim=' ', 
#                 compression='gzip',
#                 sample_size=100000  -- Smaller sample for schema detection to save memory
#             )
#         ) TO '{output_path}' (
#             FORMAT 'PARQUET', 
#             CODEC 'ZSTD',
#             ROW_GROUP_SIZE 100000  -- Smaller row groups for better memory management
#         );
#     """)
    
#     print(f"Successfully converted the entire file to {output_path}")
    
# finally:
#     # Close the connection and clean up temporary database
#     con.close()
#     if os.path.exists(temp_db_path):
#         os.remove(temp_db_path)
#         print("Cleaned up temporary database file")

File size: 189.6 GB
Available memory: 120.1 GB
Starting memory-optimized conversion with DuckDB...
Setting DuckDB memory limit to 72 GB
Using 8 threads
Converting file (this may take longer but uses less memory)...


FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

Successfully converted the entire file to <path_to_string_data>/parquet_output_duckdb/protein_links_space_delimited.parquet
Cleaned up temporary database file


In [2]:
# SYSTEM REQUIREMENTS CHECK - Run this before attempting conversion
import os
import shutil
import psutil

def check_system_requirements(input_path, output_dir):
    """Check if system has enough resources for the conversion"""
    
    print("=== SYSTEM REQUIREMENTS CHECK ===")
    
    # File size
    if os.path.exists(input_path):
        file_size_gb = os.path.getsize(input_path) / (1024**3)
        print(f"Input file size: {file_size_gb:.1f} GB")
    else:
        print(f"WARNING: Input file not found: {input_path}")
        return False
    
    # Available memory
    memory = psutil.virtual_memory()
    available_memory_gb = memory.available / (1024**3)
    total_memory_gb = memory.total / (1024**3)
    print(f"Available RAM: {available_memory_gb:.1f} GB / {total_memory_gb:.1f} GB")
    
    # Available disk space
    disk_usage = shutil.disk_usage(output_dir)
    available_disk_gb = disk_usage.free / (1024**3)
    print(f"Available disk space: {available_disk_gb:.1f} GB")
    
    # Recommendations
    print("\n=== RECOMMENDATIONS ===")
    
    # Memory recommendation
    if available_memory_gb < file_size_gb * 0.1:  # Less than 10% of file size
        print("⚠️  WARNING: Very low available memory!")
        print("   Recommended: Use the chunked processing approach")
        print("   Set chunk_size_mb to 200-500 MB")
    elif available_memory_gb < file_size_gb * 0.3:  # Less than 30% of file size
        print("⚠️  CAUTION: Limited memory available")
        print("   Recommended: Use memory-optimized DuckDB settings")
        print("   Monitor memory usage during conversion")
    else:
        print("✅ Memory should be sufficient for DuckDB conversion")
    
    # Disk space recommendation
    estimated_parquet_size = file_size_gb * 0.3  # Rough estimate: 30% of original size
    if available_disk_gb < estimated_parquet_size * 3:  # Need 3x for temp files
        print("⚠️  WARNING: May not have enough disk space!")
        print(f"   Estimated output size: {estimated_parquet_size:.1f} GB")
        print(f"   Recommended free space: {estimated_parquet_size * 3:.1f} GB")
    else:
        print("✅ Disk space should be sufficient")
    
    print(f"\n=== SUGGESTED APPROACH ===")
    if available_memory_gb >= file_size_gb * 0.3:
        print("1. Try the memory-optimized DuckDB approach first (Cell 15)")
        print("2. If it fails, use the chunked approach (Cell 16)")
    else:
        print("1. Use the chunked processing approach (Cell 16)")
        print("2. Start with chunk_size_mb=200 and increase if stable")
    
    return True

# Run the check
input_path = '<path_to_string_data>/protein.links.detailed.v12.0.txt.gz'
output_dir = '<path_to_string_data>/parquet_output_duckdb'
check_system_requirements(input_path, output_dir)


=== SYSTEM REQUIREMENTS CHECK ===
Input file size: 189.6 GB
Available RAM: 120.1 GB / 125.6 GB
Available disk space: 189.2 GB

=== RECOMMENDATIONS ===
✅ Memory should be sufficient for DuckDB conversion
✅ Disk space should be sufficient

=== SUGGESTED APPROACH ===
1. Try the memory-optimized DuckDB approach first (Cell 15)
2. If it fails, use the chunked approach (Cell 16)


True

Filter string data to only include rows with confidence >70% and which are related to a cafa5 protein

Takes ~15 minutues with 64 cpus and slow SLURM disk performance, for faster performance copy protein_links_space_delimited.parquet to SLURM_TEMPDIR with above cell

In [13]:
import polars as pl

input_parquet_path = (
    STRING_PROTEIN_LINKS_PARQUET
)

# Configuration: use all threads and a suitable streaming chunk size
pl.Config.set_streaming_chunk_size(1_000_000)  # Optional: tune chunk size for streaming

MIN_COMBINED_SCORE = 700

# Build swissprot ID list (assuming swissprot_pl is a Polars DataFrame already in memory)
string_ids = swissprot_pl["string_id"].drop_nulls().unique().to_list()
print(f"swissprot filter will use {len(string_ids)} unique string IDs")

# Lazy scan with optimized parallelism
lf = pl.scan_parquet(
    input_parquet_path,
    parallel="row_groups",   # parallelize over row groups to utilize all cores
    use_statistics=True      # enable Parquet page skipping by stats
    # low_memory=False is default (we prefer performance),
    # rechunk=False is default (avoid extra copying before collect)
)

# Apply filters lazily
lf = lf.filter(pl.col("combined_score") >= MIN_COMBINED_SCORE)
lf = lf.filter(pl.col("protein1").is_in(string_ids))

# Collect results with streaming (to pipeline reading & filtering)
print("Executing query in streaming mode...")
final_df = lf.collect(engine="streaming")  # or engine="streaming" on newer Polars

print("Done! Final dataframe has shape", final_df.shape)


swissprot filter will use 208429 unique string IDs
Executing query in streaming mode...


: 

In [7]:
# FIXED VERSION: Use the existing code with polars-u64-idx
# Run this after installing polars-u64-idx and restarting kernel

import polars as pl

input_parquet_path = (
    STRING_PROTEIN_LINKS_PARQUET
)

# Check if we have the large-dataset version of polars
try:
    # Test if polars supports large datasets
    print("Polars version:", pl.__version__)
    print("Checking polars build configuration...")
    
    # This will help identify if we have the right version
    dummy_large_range = range(2**33)  # Larger than 2^32
    print("✅ Python can handle large ranges")
    
except Exception as e:
    print(f"⚠️  Issue detected: {e}")

# Configuration: use conservative settings for large datasets
pl.Config.set_streaming_chunk_size(500_000)  # Smaller chunks for stability

MIN_COMBINED_SCORE = 700

# Build swissprot ID list
string_ids = swissprot_pl["string_id"].drop_nulls().unique().to_list()
print(f"swissprot filter will use {len(string_ids)} unique string IDs")

# Use more conservative approach for large datasets
print("Setting up lazy frame with conservative settings...")

lf = pl.scan_parquet(
    input_parquet_path,
    parallel="columns",      # Use column parallelism instead of row_groups for better memory usage
    use_statistics=True,     # Enable statistics-based filtering
    low_memory=True          # Use less memory during processing
)

# Apply filters lazily
print("Applying filters...")
lf = lf.filter(pl.col("combined_score") >= MIN_COMBINED_SCORE)
lf = lf.filter(pl.col("protein1").is_in(string_ids))

# Collect with conservative settings
print("Executing query with streaming engine...")
try:
    final_df = lf.collect(engine="streaming")
    print("✅ Success! Final dataframe has shape", final_df.shape)
except Exception as e:
    print(f"❌ Error: {e}")
    print("Try running the alternative approach in the next cell")


Polars version: 1.32.3
Checking polars build configuration...
✅ Python can handle large ranges
swissprot filter will use 208429 unique string IDs
Setting up lazy frame with conservative settings...
Applying filters...
Executing query with streaming engine...
✅ Success! Final dataframe has shape (6760068, 10)


In [8]:
final_df.head(10)

protein1,protein2,neighborhood,fusion,cooccurence,coexpression,experimental,database,textmining,combined_score
str,str,i64,i64,i64,i64,i64,i64,i64,i64
"""23.BEL05_04305""","""23.BEL05_18805""",116,0,734,259,0,0,372,875
"""23.BEL05_04305""","""23.BEL05_04310""",569,0,729,813,99,952,531,999
"""23.BEL05_04305""","""23.BEL05_07380""",116,0,181,712,0,435,173,884
"""23.BEL05_04305""","""23.BEL05_04080""",109,0,0,0,0,907,173,925
"""23.BEL05_04305""","""23.BEL05_18800""",116,0,754,250,0,0,373,884
"""23.BEL05_04305""","""23.BEL05_00825""",264,0,0,559,0,484,385,883
"""23.BEL05_04305""","""23.BEL05_05665""",92,0,0,419,0,952,374,981
"""23.BEL05_04305""","""23.BEL05_18815""",116,0,0,615,0,0,381,770
"""23.BEL05_04305""","""23.BEL05_04765""",92,0,0,419,0,952,379,981


In [10]:
print("Estimated in-memory size:", final_df.estimated_size() / 1e6, "MB")

Estimated in-memory size: 644.104237 MB


Writes the filtered string dataframe to disk

In [11]:
output_path = os.path.join(OUTPUT_DIR, "filtered_protein_links.parquet")
final_df.write_parquet(output_path, compression="zstd")


Reads the filtered string dataframe

In [12]:
import polars as pl

# Path to the saved file
output_path = os.path.join(OUTPUT_DIR, "filtered_protein_links.parquet")

# Read the file
filtered_df = pl.read_parquet(output_path)

# Show shape and head
print("Shape:", filtered_df.shape)
print(filtered_df.tail())


Shape: (6760068, 10)
shape: (5, 10)
┌────────────┬────────────┬────────────┬────────┬───┬───────────┬──────────┬───────────┬───────────┐
│ protein1   ┆ protein2   ┆ neighborho ┆ fusion ┆ … ┆ experimen ┆ database ┆ textminin ┆ combined_ │
│ ---        ┆ ---        ┆ od         ┆ ---    ┆   ┆ tal       ┆ ---      ┆ g         ┆ score     │
│ str        ┆ str        ┆ ---        ┆ i64    ┆   ┆ ---       ┆ i64      ┆ ---       ┆ ---       │
│            ┆            ┆ i64        ┆        ┆   ┆ i64       ┆          ┆ i64       ┆ i64       │
╞════════════╪════════════╪════════════╪════════╪═══╪═══════════╪══════════╪═══════════╪═══════════╡
│ 1921421.M4 ┆ 1921421.M4 ┆ 55         ┆ 0      ┆ … ┆ 0         ┆ 800      ┆ 0         ┆ 802       │
│ 93_12200   ┆ 93_06140   ┆            ┆        ┆   ┆           ┆          ┆           ┆           │
│ 1921421.M4 ┆ 1921421.M4 ┆ 0          ┆ 0      ┆ … ┆ 0         ┆ 900      ┆ 0         ┆ 900       │
│ 93_12200   ┆ 93_14215   ┆            ┆        ┆   ┆  

replace protein2 string code with protein2 name

In [13]:
import polars as pl

# --- Replace protein2 IDs with their preferred names ---
# 1. Rename original protein2 to protein2_string
filtered_df = filtered_df.rename({"protein2": "protein2_string"})

# 2. Join to bring in the preferred_name from protein_info_df
filtered_df = filtered_df.join(
    protein_info_df.select(["string_protein_id", "preferred_name"]),
    left_on="protein2_string",
    right_on="string_protein_id",
    how="left"
)

# 3. Overwrite protein2 with the joined preferred_name
filtered_df = filtered_df.with_columns(
    pl.col("preferred_name").alias("protein2")
)

# 4. Drop any helper columns that were added
to_drop = [c for c in ["string_protein_id", "preferred_name"] if c in filtered_df.columns]
filtered_df = filtered_df.drop(to_drop)

# Now filtered_df.protein2 contains the preferred names,
# and the original IDs are in filtered_df.protein2_string.


In [14]:
filtered_df

protein1,protein2_string,neighborhood,fusion,cooccurence,coexpression,experimental,database,textmining,combined_score,protein2
str,str,i64,i64,i64,i64,i64,i64,i64,i64,str
"""23.BEL05_04305""","""23.BEL05_18805""",116,0,734,259,0,0,372,875,"""OEG74489.1"""
"""23.BEL05_04305""","""23.BEL05_04310""",569,0,729,813,99,952,531,999,"""OEG72217.1"""
"""23.BEL05_04305""","""23.BEL05_07380""",116,0,181,712,0,435,173,884,"""OEG75425.1"""
"""23.BEL05_04305""","""23.BEL05_04080""",109,0,0,0,0,907,173,925,"""hisC"""
"""23.BEL05_04305""","""23.BEL05_18800""",116,0,754,250,0,0,373,884,"""OEG74488.1"""
…,…,…,…,…,…,…,…,…,…,…
"""1921421.M493_12200""","""1921421.M493_06140""",55,0,0,0,0,800,0,802,"""sucC"""
"""1921421.M493_12200""","""1921421.M493_14215""",0,0,0,0,0,900,0,900,"""accA"""
"""1921421.M493_12200""","""1921421.M493_16565""",67,0,0,0,0,900,61,904,"""AGT33525.1"""


In [15]:
print("swissprot column types:")
print(swissprot_pl.schema)

print("\nFiltered DF column types:")
print(filtered_df.schema)


swissprot column types:
Schema([('protein_id', String), ('protein_names', String), ('protein_function', String), ('organism', String), ('length', Int64), ('subcellular_location', String), ('sequence', String), ('go_ids', List(String)), ('go_bp', List(String)), ('go_mf', List(String)), ('go_cc', List(String)), ('structure_path', String), ('interpro_ids', List(String)), ('interpro_location', String), ('string_id', String)])

Filtered DF column types:
Schema([('protein1', String), ('protein2_string', String), ('neighborhood', Int64), ('fusion', Int64), ('cooccurence', Int64), ('coexpression', Int64), ('experimental', Int64), ('database', Int64), ('textmining', Int64), ('combined_score', Int64), ('protein2', String)])


Groups filtered string df by protein 1 and then adds column with all associated protein 2 and another with all proteins 2 along with individual scores

In [16]:
import polars as pl

grouped_df = (
    filtered_df
    .group_by("protein1")
    .agg([
        # 1. List of all protein2 values per protein1
        pl.col("protein2").implode().alias("interaction_partners"),

        # 2. Struct of full interaction info, collected into list
        pl.struct([
            pl.col("protein2"),
            pl.col("neighborhood"),
            pl.col("fusion"),
            pl.col("cooccurence"),
            pl.col("coexpression"),
            pl.col("experimental"),
            pl.col("database"),
            pl.col("textmining"),
            pl.col("combined_score"),
        ]).implode().alias("full_interaction_info"),
    ])
)

print("Grouped shape:", grouped_df.shape)
print(grouped_df.head())


Grouped shape: (196977, 3)
shape: (5, 3)
┌────────────────────┬─────────────────────────────────┬─────────────────────────────────┐
│ protein1           ┆ interaction_partners            ┆ full_interaction_info           │
│ ---                ┆ ---                             ┆ ---                             │
│ str                ┆ list[str]                       ┆ list[struct[9]]                 │
╞════════════════════╪═════════════════════════════════╪═════════════════════════════════╡
│ 272844.PAB1184     ┆ ["PAB1179", "atpE", … "atpB"]   ┆ [{"PAB1179",794,0,0,157,498,90… │
│ 315749.Bcer98_0297 ┆ ["gatB", "aspS", … "rpsZ"]      ┆ [{"gatB",743,0,713,732,860,900… │
│ 259536.Psyc_0390   ┆ ["rsfS"]                        ┆ [{"rsfS",109,0,160,752,0,0,106… │
│ 74546.PMT9312_0519 ┆ ["ABB49456.1", "ABB49785.1", …… ┆ [{"ABB49456.1",104,0,0,0,0,900… │
│ 167546.P9301_08341 ┆ ["ABO17458.1", "panB", … "eda"… ┆ [{"ABO17458.1",774,0,0,0,0,0,0… │
└────────────────────┴───────────────────────────

In [18]:
swissprot_pl

protein_id,protein_names,protein_function,organism,length,subcellular_location,sequence,go_ids,go_bp,go_mf,go_cc,structure_path,interpro_ids,interpro_location,string_id
str,str,str,str,i64,str,str,list[str],list[str],list[str],list[str],str,list[str],str,str
"""Q8CX00""","""Chaperonin GroEL""","""Together with its co-chaperoni…","""Streptococcus agalactiae serot…",540,"""Cytoplasm""","""MAKDIKFSADARSAMVRGVDILADTVKVTL…","[""GO:0008150"", ""GO:0005575"", … ""GO:0005524""]","[""GO:0008150"", ""GO:0009987"", … ""GO:0042026""]","[""GO:0003674"", ""GO:0005488"", … ""GO:0005524""]","[""GO:0005575"", ""GO:0110165"", ""GO:0005737""]","""AF-Q8CX00-F1-model_v4.cif""","[""IPR018370"", ""IPR001844"", … ""IPR027409""]","""{""IPR027413"": [5, 519], ""IPR02…","""208435.SAG2074"""
"""Q4P9K9""","""Chitin synthase 8""","""Polymerizes chitin, a structur…","""Ustilago maydis (strain 521 / …",2005,"""Cell membrane; Cytoplasmic ves…","""MSALDEVAKLSQLTNITPDTIFSVLRDRFY…","[""GO:0008150"", ""GO:0005575"", … ""GO:0005524""]","[""GO:0008150"", ""GO:0009987"", … ""GO:0071555""]","[""GO:0003674"", ""GO:0003774"", … ""GO:0005524""]","[""GO:0005575"", ""GO:0110165"", … ""GO:0031410""]","""AF-Q4P9K9-F1-model_v4.cif""","[""IPR001609"", ""IPR004835"", … ""IPR036037""]","""{""IPR027417"": [5, 771], ""IPR03…","""237631.Q4P9K9"""
"""A9WUW1""","""ATP-dependent Clp protease ATP…","""ATP-dependent specificity comp…","""Renibacterium salmoninarum (st…",427,null,"""MARMGESTDLLKCSFCGKSQKQVRKLIAGP…","[""GO:0008150"", ""GO:0003674"", … ""GO:0005524""]","[""GO:0008150"", ""GO:0009987"", ""GO:0006457""]","[""GO:0003674"", ""GO:0005488"", … ""GO:0005524""]",null,"""AF-A9WUW1-F1-model_v4.cif""","[""IPR050052"", ""IPR038366"", … ""IPR019489""]","""{""IPR038366"": [3, 50], ""IPR027…","""288705.RSal33209_3266"""
"""A0A411KZU9""","""Burnettramic acids biosynthesi…","""Part of the gene cluster that …","""Aspergillus burnettii""",412,null,"""MAIASSIGLVAEAIHRHKTPERPIESENDI…","[""GO:0008150"", ""GO:0008152"", … ""GO:0017000""]","[""GO:0008150"", ""GO:0008152"", … ""GO:0017000""]",null,null,"""AF-A0A411KZU9-F1-model_v4.cif""","[""IPR053221""]","""{""IPR053221"": [2, 363]}""",null
"""Q0AE59""","""50S ribosomal protein L34""",null,"""Nitrosomonas eutropha (strain …",44,null,"""MKRTYQPSVISKKRTHGFRARMKTRGGRAV…","[""GO:0008150"", ""GO:0005575"", … ""GO:0006412""]","[""GO:0008150"", ""GO:0008152"", … ""GO:0006412""]","[""GO:0003674"", ""GO:0005198"", ""GO:0003735""]","[""GO:0005575"", ""GO:0032991"", … ""GO:0005840""]","""AF-Q0AE59-F1-model_v4.cif""","[""IPR020939"", ""IPR000271""]","""{""IPR000271"": [1, 44], ""IPR020…","""335283.Neut_2151"""
…,…,…,…,…,…,…,…,…,…,…,…,…,…,…
"""Q0C5R4""","""3-phosphoshikimate 1-carboxyvi…","""Catalyzes the transfer of the …","""Hyphomonas neptunium (strain A…",439,"""Cytoplasm""","""MVWTSHPVKRLAGAIRAPGDKSCSHRALIF…","[""GO:0008150"", ""GO:0005575"", … ""GO:0009073""]","[""GO:0008150"", ""GO:0008152"", … ""GO:0009073""]","[""GO:0003674"", ""GO:0003824"", … ""GO:0003866""]","[""GO:0005575"", ""GO:0110165"", ""GO:0005737""]","""AF-Q0C5R4-F1-model_v4.cif""","[""IPR013792"", ""IPR006264"", … ""IPR036968""]","""{""IPR013792"": [5, 436], ""IPR03…","""228405.HNE_0195"""
"""P0A6S8""","""Glycerol-3-phosphate dehydroge…",null,"""Escherichia coli O6:H1 (strain…",339,"""Cytoplasm""","""MNQRNASMTVIGAGSYGTALAITLARNGHE…","[""GO:0008150"", ""GO:0005575"", … ""GO:0046168""]","[""GO:0008150"", ""GO:0008152"", … ""GO:0046168""]","[""GO:0003674"", ""GO:0005488"", … ""GO:0051287""]","[""GO:0005575"", ""GO:0032991"", … ""GO:0009331""]","""AF-P0A6S8-F1-model_v4.cif""","[""IPR036291"", ""IPR013328"", … ""IPR011128""]","""{""IPR036291"": [8, 182], ""IPR00…","""199310.c4430"""
"""B7NN18""","""Periplasmic nitrate reductase""","""Catalytic subunit of the perip…","""Escherichia coli O7:K1 (strain…",828,"""Periplasm""","""MKLSRRSFMKANAVAAAAAAAGLSVPGVAR…","[""GO:0008150"", ""GO:0005575"", … ""GO:0006777""]","[""GO:0008150"", ""GO:0008152"", … ""GO:0006777""]","[""GO:0003674"", ""GO:0005488""

Connects the PPI data with the cafa5 data

In [19]:
merged_df = swissprot_pl.join(
    grouped_df,
    left_on="string_id",
    right_on="protein1",
    how="left"
)

# Only drop 'protein1' if it exists (e.g., when join didn't auto-drop it)
if "protein1" in merged_df.columns:
    merged_df = merged_df.drop("protein1")

print("Merged shape:", merged_df.shape)
print(merged_df.select(["string_id", "interaction_partners", "full_interaction_info"]).head())


Merged shape: (419049, 17)
shape: (5, 3)
┌───────────────────────┬─────────────────────────────────┬─────────────────────────────────┐
│ string_id             ┆ interaction_partners            ┆ full_interaction_info           │
│ ---                   ┆ ---                             ┆ ---                             │
│ str                   ┆ list[str]                       ┆ list[struct[9]]                 │
╞═══════════════════════╪═════════════════════════════════╪═════════════════════════════════╡
│ 208435.SAG2074        ┆ ["groES", "dnaJ", … "fusA"]     ┆ [{"groES",588,0,294,924,999,0,… │
│ 237631.Q4P9K9         ┆ ["UMAG_11497"]                  ┆ [{"UMAG_11497",0,0,0,103,0,912… │
│ 288705.RSal33209_3266 ┆ ["atpD", "clpP.3", … "dnaK"]    ┆ [{"atpD",0,814,0,0,0,0,246,854… │
│ null                  ┆ null                            ┆ null                            │
│ 335283.Neut_2151      ┆ ["Neut_1653", "nusA", … "rplS"… ┆ [{"Neut_1653",0,0,0,107,809,0,… │
└──────────────────

In [22]:
# The percentage of rows in swissprot WITHOUT an entry in string->uniprot table
sum(merged_df['string_id'].is_null())

210620

In [23]:
# The percent of rows in swissprot WITHOUT interaction partners
sum(merged_df['interaction_partners'].is_null())/len(merged_df)*100

52.99427990521395

Saves the cafa5 data with the 2 new PPI columns to disk

In [24]:
output_path = os.path.join(OUTPUT_DIR, "swissprot_merged_with_interactions.parquet")

merged_df.write_parquet(output_path, compression="zstd")


In [25]:
import polars as pl

# Path to the saved file
merged_path = os.path.join(OUTPUT_DIR, "swissprot_merged_with_interactions.parquet")

# Read the merged DataFrame
merged_df = pl.read_parquet(merged_path)

# Preview shape and a few rows
print("Shape:", merged_df.shape)
merged_df.sample(9)

Shape: (419049, 17)


protein_id,protein_names,protein_function,organism,length,subcellular_location,sequence,go_ids,go_bp,go_mf,go_cc,structure_path,interpro_ids,interpro_location,string_id,interaction_partners,full_interaction_info
str,str,str,str,i64,str,str,list[str],list[str],list[str],list[str],str,list[str],str,str,list[str],list[struct[9]]
"""Q31YW2""","""S-formylglutathione hydrolase …","""Serine hydrolase involved in t…","""Shigella boydii serotype 4 (st…",278,null,"""MEMLEEHRCFEGWQQRWRHDSSTLNCPMTF…","[""GO:0008150"", ""GO:0003674"", … ""GO:0046294""]","[""GO:0008150"", ""GO:0008152"", … ""GO:0046294""]","[""GO:0003674"", ""GO:0003824"", … ""GO:0018738""]",null,"""AF-Q31YW2-F1-model_v4.cif""","[""IPR014186"", ""IPR029058"", ""IPR000801""]","""{""IPR029058"": [1, 277], ""IPR01…",null,null,null
"""Q8TZB2""","""50S ribosomal protein L34e""",null,"""Methanopyrus kandleri (strain …",113,null,"""MPAPRYRSRSCRRVYKRTPGGRTVIHFEKK…","[""GO:0008150"", ""GO:0005575"", … ""GO:0006412""]","[""GO:0008150"", ""GO:0008152"", … ""GO:0006412""]","[""GO:0003674"", ""GO:0005198"", ""GO:0003735""]","[""GO:0005575"", ""GO:0032991"", … ""GO:0005840""]","""AF-Q8TZB2-F1-model_v4.cif""","[""IPR038562"", ""IPR018065"", … ""IPR008195""]","""{""IPR038562"": [32, 96], ""IPR04…","""190192.MK0023""","[""MK1619"", ""MK0125"", … ""rpsG""]","[{""MK1619"",0,0,152,772,0,0,0,798}, {""MK0125"",0,0,0,999,999,0,221,999}, … {""rpsG"",63,0,0,989,999,0,265,999}]"
"""Q8D209""","""50S ribosomal protein L2""","""One of the primary rRNA bindin…","""Wigglesworthia glossinidia bre…",275,null,"""MTLNKCNPTTPSRRHTVKVVNNKLYKGKSF…","[""GO:0008150"", ""GO:0005575"", … ""GO:0006412""]","[""GO:0008150"", ""GO:0008152"", … ""GO:0006412""]","[""GO:0003674"", ""GO:0003824"", … ""GO:0019843""]","[""GO:0005575"", ""GO:0110165"", … ""GO:0005840""]","""AF-Q8D209-F1-model_v4.cif""","[""IPR022669"", ""IPR008991"", … ""IPR005880""]","""{""IPR012340"": [1, 125], ""IPR01…","""36870.gene:10369055""","[""rpsP"", ""yidC"", … ""rpmE""]","[{""rpsP"",0,0,0,939,910,0,412,996}, {""yidC"",0,0,0,612,401,0,0,757}, … {""rpmE"",0,0,0,179,910,0,235,938}]"
"""P37352""","""Homoprotocatechuate catabolism…","""Decarboxylates OPET (5-oxo-pen…","""Escherichia coli""",427,null,"""MKGTIFAVALNHRSQLDAWQEAFQQSPIKA…","[""GO:0008150"", ""GO:0003674"", … ""GO:1901023""]","[""GO:0008150"", ""GO:0008152"", … ""GO:1901023""]","[""GO:0003674"", ""GO:0005488"", … ""GO:0018800""]",null,"""AF-P37352-F1-model_v4.cif""","[""IPR011234"", ""IPR036663"", … ""IPR012684""]","""{""IPR036663"": [1, 427], ""IPR01…",null,null,null
"""O21872""","""Probable portal protein""","""Forms the portal vertex of the…","""Lactococcus phage SK1""",378,"""Virion""","""MNLFGKVVSFSRGKLNNDTQRVTAWQNEAV…","[""GO:0008150"", ""GO:0005575"", … ""GO:0099001""]","[""GO:0008150"", ""GO:0016032"", … ""GO:0099001""]",null,"[""GO:0005575"", ""GO:0044423"", ""GO:0019028""]",null,null,"""NaN""",null,null,null
"""Q1RC21""","""tRNA-cytidine(32) 2-sulfurtran…","""Catalyzes the ATP-dependent 2-…","""Escherichia coli (strain UTI89…",311,"""Cytoplasm""","""MQENQQITKKEQYNLNKLQKRLRRNVGEAI…","[""GO:0008150"", ""GO:0005575"", … ""GO:0034227""]","[""GO:0008150"", ""GO:0008152"", … ""GO:0034227""]","[""GO:0003674"", ""GO:0005488"", … ""GO:0005524""]","[""GO:0005575"", ""GO:0110165"", ""GO:0005737""]","""AF-Q1RC21-F1-model_v4.cif""","[""IPR014729"", ""IPR035107"", … ""IPR012089""]","""{""IPR014729"": [2, 264], ""IPR03…",null,null,null
"""P44048""","""Probable Fe(2+)-trafficking pr…","""Could be a mediator in iron tr…","""Haemophilus influenzae (strain…",90,null,"""MARTVFCEYLKKEAEGLDFQLYPGELGKRI…","[""GO:0003674"", ""GO:0005488"", … ""GO:0005506""]",null,"[""GO:0003674"", ""GO:0005488"", … ""GO:0005506""]",null,"""AF-P44048-F1-model_v4.cif""","[""IPR007457"", ""IPR036766""]","""{""IPR036766"": [1, 90], ""IPR007…","""71421.HI_0760""","[""mltC"", ""mutY""]","[{""mltC"",760,0,0,75,0,0,0,768}, {""mutY"",793,0,0,59,0,0,0,797}]"
"""Q0CRX0""","""Tyrosinase P""","""Tyrosinase; part of the gene c…","""Asp

In [26]:
merged_df.sample(10)

protein_id,protein_names,protein_function,organism,length,subcellular_location,sequence,go_ids,go_bp,go_mf,go_cc,structure_path,interpro_ids,interpro_location,string_id,interaction_partners,full_interaction_info
str,str,str,str,i64,str,str,list[str],list[str],list[str],list[str],str,list[str],str,str,list[str],list[struct[9]]
"""A0LDW8""","""ATP synthase subunit b""","""F(1)F(0) ATP synthase produces…","""Magnetococcus marinus (strain …",189,"""Cell inner membrane""","""MISAAYAATHAAAEHAQSGMPQFDSSTFSS…","[""GO:0008150"", ""GO:0005575"", … ""GO:0015986""]","[""GO:0008150"", ""GO:0008152"", … ""GO:0015986""]","[""GO:0003674"", ""GO:0003824"", … ""GO:0046933""]","[""GO:0005575"", ""GO:0110165"", … ""GO:0045263""]","""AF-A0LDW8-F1-model_v4.cif""","[""IPR002146"", ""IPR005864"", ""IPR050059""]","""{""IPR050059"": [20, 180], ""IPR0…","""156889.Mmc1_3676""","[""rplW"", ""rpoA"", … ""atpC""]","[{""rplW"",0,0,0,846,0,0,141,862}, {""rpoA"",0,0,0,844,0,0,239,876}, … {""atpC"",134,0,0,929,927,900,289,999}]"
"""A9L5P8""","""50S ribosomal protein L19""","""This protein is located at the…","""Shewanella baltica (strain OS1…",117,null,"""MNNIIKMLNDEQMKQDVPAFGAGDTVVVQV…","[""GO:0008150"", ""GO:0005575"", … ""GO:0006412""]","[""GO:0008150"", ""GO:0008152"", … ""GO:0006412""]","[""GO:0003674"", ""GO:0005198"", ""GO:0003735""]","[""GO:0005575"", ""GO:0032991"", … ""GO:0005840""]","""AF-A9L5P8-F1-model_v4.cif""","[""IPR038657"", ""IPR001857"", … ""IPR018257""]","""{""IPR008991"": [1, 114], ""IPR03…",null,null,null
"""Q5M1X5""","""30S ribosomal protein S2""",null,"""Streptococcus thermophilus (st…",255,null,"""MAVISMKQLLEAGVHFGHQTRRWNPKMAKY…","[""GO:0008150"", ""GO:0005575"", … ""GO:0006412""]","[""GO:0008150"", ""GO:0008152"", … ""GO:0006412""]","[""GO:0003674"", ""GO:0005198"", ""GO:0003735""]","[""GO:0005575"", ""GO:0110165"", … ""GO:0005840""]","""AF-Q5M1X5-F1-model_v4.cif""","[""IPR023591"", ""IPR018130"", … ""IPR005706""]","""{""IPR023591"": [6, 230], ""IPR00…",null,null,null
"""B5XRE3""","""DNA replication terminus site-…","""Trans-acting protein required …","""Klebsiella pneumoniae (strain …",310,"""Cytoplasm""","""MASYDLVERLNDTFRQIELELQTLQQALSS…","[""GO:0008150"", ""GO:0005575"", … ""GO:0006274""]","[""GO:0008150"", ""GO:0008152"", … ""GO:0006260""]","[""GO:0003674"", ""GO:0005488"", … ""GO:0003677""]","[""GO:0005575"", ""GO:0110165"", ""GO:0005737""]","""AF-B5XRE3-F1-model_v4.cif""","[""IPR036381"", ""IPR036384"", ""IPR008865""]","""{""IPR036384"": [5, 307], ""IPR03…",null,null,null
"""B9DSW1""","""50S ribosomal protein L24""","""One of two assembly initiator …","""Streptococcus uberis (strain A…",101,null,"""MFVKKGDKVRVIAGKDKGTEAVVLKALPKV…","[""GO:0008150"", ""GO:0005575"", … ""GO:0006412""]","[""GO:0008150"", ""GO:0008152"", … ""GO:0006412""]","[""GO:0003674"", ""GO:0005488"", … ""GO:0019843""]","[""GO:0005575"", ""GO:0032991"", … ""GO:0005840""]","""AF-B9DSW1-F1-model_v4.cif""","[""IPR005824"", ""IPR008991"", … ""IPR057264""]","""{""IPR014722"": [1, 99], ""IPR008…","""218495.SUB0079""","[""rpsN"", ""rpsL"", … ""ffh""]","[{""rpsN"",702,0,0,601,642,81,176,961}, {""rpsL"",158,0,0,771,933,0,254,989}, … {""ffh"",45,0,0,112,932,82,118,944}]"
"""Q5HQ36""","""Phenylalanine--tRNA ligase alp…",null,"""Staphylococcus epidermidis (st…",352,"""Cytoplasm""","""MTQNDLMSELKQQALVDINEANDAQALQEV…","[""GO:0008150"", ""GO:0005575"", … ""GO:0006432""]","[""GO:0008150"", ""GO:0008152"", … ""GO:0006432""]","[""GO:0003674"", ""GO:0005488"", … ""GO:0005524""]","[""GO:0005575"", ""GO:0110165"", ""GO:0005737""]","""AF-Q5HQ36-F1-model_v4.cif""","[""IPR010978"", ""IPR002319"", … ""IPR022911""]","""{""IPR045864"": [10, 346], ""IPR0…","""176279.SERP0721""","[""tyrS"", ""pheT"", … ""fusA""]","[{""tyrS"",109,0,0,447,0,0,456,708}, {""pheT"",883,0,713,959,986,933,415,999}, … {""fusA"",46,0,282,487,114,0,271,731}]"
"""A4Y564""","""Probable phosphatase Sputcn32_…",null,"""Shewanella putrefaciens (strai…",251,null,"""MQYQVDTHTHTVASTHAYSTIHDYLAVAKQ…","[""GO:0003674"", ""GO:00

In [30]:
from datasets import Dataset
import polars as pl

# Extract protein IDs from existing train and test DataFrames
train_protein_ids = train_pl["protein_id"].unique().to_list()
test_protein_ids = test_pl["protein_id"].unique().to_list()

print(f"Train set has {len(train_protein_ids)} unique protein IDs")
print(f"Test set has {len(test_protein_ids)} unique protein IDs")

# Check for any overlap between train and test (should be none)
overlap = set(train_protein_ids) & set(test_protein_ids)
if overlap:
    print(f"⚠️  WARNING: Found {len(overlap)} overlapping protein IDs between train and test")
else:
    print("✅ No overlap between train and test protein IDs")

# Split merged_df using protein_id matching
train_merged = merged_df.filter(pl.col("protein_id").is_in(train_protein_ids))
test_merged = merged_df.filter(pl.col("protein_id").is_in(test_protein_ids))

print(f"Train merged dataset: {train_merged.shape}")
print(f"Test merged dataset: {test_merged.shape}")
print(f"Total merged dataset: {merged_df.shape}")

# Verify we captured all data
total_split = train_merged.height + test_merged.height
if total_split == merged_df.height:
    print("✅ All data successfully split into train/test")
else:
    missing = merged_df.height - total_split
    print(f"⚠️  {missing} rows not assigned to train or test (possibly due to missing protein IDs)")

# Convert to pandas for HuggingFace Dataset creation
train_pd = train_merged.to_pandas()
test_pd = test_merged.to_pandas()

# Create HuggingFace datasets
train_hf = Dataset.from_pandas(train_pd, preserve_index=False)
test_hf = Dataset.from_pandas(test_pd, preserve_index=False)

print(f"\nHuggingFace datasets created:")
print(f"Train: {len(train_hf)} samples")
print(f"Test: {len(test_hf)} samples")


Train set has 333677 unique protein IDs
Test set has 83420 unique protein IDs
✅ No overlap between train and test protein IDs
Train merged dataset: (335230, 17)
Test merged dataset: (83819, 17)
Total merged dataset: (419049, 17)
✅ All data successfully split into train/test

HuggingFace datasets created:
Train: 335230 samples
Test: 83819 samples


In [31]:
train_hf

Dataset({
    features: ['protein_id', 'protein_names', 'protein_function', 'organism', 'length', 'subcellular_location', 'sequence', 'go_ids', 'go_bp', 'go_mf', 'go_cc', 'structure_path', 'interpro_ids', 'interpro_location', 'string_id', 'interaction_partners', 'full_interaction_info'],
    num_rows: 335230
})

In [34]:
# Upload dataset to Hugging Face Hub
# First, login to Hugging Face

from huggingface_hub import login, HfApi
import os

# Login to Hugging Face (this will prompt for your token)
print("Please login to Hugging Face Hub...")
print("You'll need your Hugging Face token (get it from https://huggingface.co/settings/tokens)")

try:
    login()
    print("✅ Successfully logged in to Hugging Face!")
except Exception as e:
    print(f"❌ Login failed: {e}")
    print("Please make sure you have a valid token and try again.")


Please login to Hugging Face Hub...
You'll need your Hugging Face token (get it from https://huggingface.co/settings/tokens)


✅ Successfully logged in to Hugging Face!


In [35]:
# Upload the dataset to Hugging Face Hub
# This will create a new dataset repo and upload your data

from datasets import DatasetDict

# Configuration for upload
REPO_NAME = "wanglab/cafa5"  # Change this to your desired repo name
# Alternative: use your username like "yourusername/cafa5_with_ppi"

print(f"Preparing to upload dataset to: {REPO_NAME}")
print("This may take several minutes depending on dataset size...")

# First, create the DatasetDict from your train_hf and test_hf
print("Creating DatasetDict from train and test datasets...")
cafa5_ds = DatasetDict({
    "train": train_hf,
    "test": test_hf,
})

print(f"DatasetDict created with {len(cafa5_ds['train'])} training and {len(cafa5_ds['test'])} test samples")

try:
    # Push the dataset to the Hub with task_name configuration
    print("Uploading to Hugging Face Hub...")
    cafa5_ds.push_to_hub(
        repo_id=REPO_NAME,
        config_name="swissprot_reasoning",  # This sets the task_name/config_name
        private=False,  # Set to True if you want a private dataset
        commit_message="Add CAFA5 dataset with protein-protein interactions from STRING database"
    )
    
    print(f"✅ Successfully uploaded dataset to: https://huggingface.co/datasets/{REPO_NAME}")
    print(f"Dataset contains {len(cafa5_ds['train'])} training samples and {len(cafa5_ds['test'])} test samples")
    print(f"Task name/config: swissprot_reasoning")
    
except Exception as e:
    print(f"❌ Upload failed: {e}")
    print("Common issues:")
    print("1. Make sure you're logged in (run previous cell)")
    print("2. Check if the repository name already exists")
    print("3. Verify you have permission to create datasets under 'wanglab' org")
    print("   - If not, change REPO_NAME to use your username instead")


Preparing to upload dataset to: wanglab/cafa5
This may take several minutes depending on dataset size...
Creating DatasetDict from train and test datasets...
DatasetDict created with 335230 training and 83819 test samples
Uploading to Hugging Face Hub...


Uploading the dataset shards:   0%|          | 0/3 [00:00<?, ?it/s]

Creating parquet from Arrow format:   0%|          | 0/112 [00:00<?, ?ba/s]

Uploading files as a binary IO buffer is not supported by Xet Storage. Falling back to HTTP upload.


Creating parquet from Arrow format:   0%|          | 0/112 [00:00<?, ?ba/s]

...

Uploading the dataset shards:   0%|          | 0/1 [00:00<?, ?it/s]

Creating parquet from Arrow format:   0%|          | 0/84 [00:00<?, ?ba/s]

Uploading files as a binary IO buffer is not supported by Xet Storage. Falling back to HTTP upload.


README.md: 0.00B [00:00, ?B/s]

✅ Successfully uploaded dataset to: https://huggingface.co/datasets/wanglab/cafa5
Dataset contains 335230 training samples and 83819 test samples
Task name/config: swissprot_reasoning
